Bill of Quantities - PV Toolestown II
=
This notebook counts the fixed-tilt structures and measures the trenches, ducts and cables drawn in the PV Toolestown II BOQ drawing. Lengths are summed **per layer** and written to `C:\Users\Usuario\Desktop\Toolestown II - BOQ.xlsx`. The layer name tells the item type and, through its suffix, the number of circuits (`NNxC`) or conduits (`NS`).

This is Toolestown II (PCS 06 to 11, layers `EMF_em_PCS_<nn>`). `BOQ - Toolestown (1-5).ipynb` is the same notebook for Toolestown I. Both read from the same `dwg_folder`.

**Workflow overview:**
1. Convert the DWG to DXF and load all model space entities into `pv_projects.duckdb` (project `"PV Toolestown II"`).
2. Select the entities of the project as a DuckDB relation (`data`).
3. **Structures:** take every string block, group the blocks into east-west rows (same Y) and split each row where there is a gap in X. The number of groups is the number of structures (printed only, not exported).
4. **Cables:** select AC trenches, AC access ductbanks, DC cables, DC ducts and DC trenches by layer name, then sum `length` per layer and count the pieces.
5. Write every summary table to its own sheet and close the connection.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Toolestown II - BOQ.dwg` in the project's `01 - Layout` folder inside the synced Ecotechnee `Office - Documentos` folder (`dwg_folder`). The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** blocks and lines inside other blocks, xrefs or paper space aren't read.
- **String blocks:**
  - One block reference (`INSERT`) per string, named exactly `EMF422-2_string_tilt_15°` (exact match, case-sensitive, including the `°`).
  - Dynamic block references whose properties were changed get an anonymous name (`*U<n>`) and are **not** counted. `BOQ - Reask.ipynb` has to add `*U2` for this reason; check the block names in `data` for `*U` names (e.g. `SELECT name, COUNT(*) FROM data GROUP BY name`).
  - The structures run east-west: all the strings of one table row have the same insertion Y (rounded to the metre), and consecutive strings of a structure are at most 13 m apart in X (the pitch is ~11.7 m).
  - Two structures in the same row must be more than 13 m apart in X, otherwise they are counted as one.
- **Entity types:** trenches, ducts and cables must be `LINE` or `LWPOLYLINE` (only these get a `length`). Drawing units are metres.
- **Layer names** (case-sensitive):
  - AC trenches: `LIKE 'EMF_bt_vala_btca_%xC'`.
  - AC ducts in access crossings: `LIKE 'EMF_bt_access_ductbank_%xC'`.
  - DC cables: exactly `EMF_cabo_solar_negativo` and `EMF_cabo_solar_positivo`.
  - DC ducts: `LIKE 'EMF_bt_eletroduto_%S'`.
  - DC trenches: exactly `EMF_DC_vala` (a single layer, so there is no split by number of circuits).
  - In `LIKE`, `%` = any text and `_` also matches any single character.
- **Ducts and crossings are counted per entity:** each duct piece / crossing must be one single LINE or LWPOLYLINE.

Extracting the AutoCAD data and loading it into the DuckDB database
=
- Copies `PV Toolestown II - BOQ.dwg` from `dwg_folder` into `temp_folder`, so the ODA File Converter only converts that one file and not every DWG in the project folder.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) writes the DXF (ACAD2018) into `output_folder`, which is the same folder as the DWG.
- `load_to_duckdb` reads every model space entity with `ezdxf` and loads it into the `cad_entities` table under the project name `"PV Toolestown II"`. If the project already exists, its old rows are deleted first, so the table always holds the latest version of the drawing.
  - `LINE` / `LWPOLYLINE`: start/end point and length (for polylines, along all segments including bulge arcs).
  - `INSERT` (block reference): block name, insertion point (`X`, `Y`), rotation, color and the attribute values.
- At the end, the temporary DXF and the copied DWG are deleted. The DuckDB connection (`conn`) stays open for the rest of the notebook and is closed in the last cell.

In [15]:
from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\2 - Toolestown Solar\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout"
temp_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\2 - Toolestown Solar\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Toolestown II - BOQ.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Toolestown II - BOQ.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Toolestown II - BOQ.dwg")

conn = load_to_duckdb(dxf_file, "PV Toolestown II")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Toolestown II already exists
Removed existing data for project: PV Toolestown II
Successfully loaded project
Loaded 8400 entities for project: PV Toolestown II
Don't forget to close connection.
Data Loaded
DuckDB Loaded


CABLE AND STRUCTURE DATA
=

- `data` is a DuckDB relation with `name`, `layer`, insertion point (`X`, `Y`), start/end coordinates and `length` of project `"PV Toolestown II"`. The next cells query it by its Python variable name (DuckDB can query Python relations/DataFrames by name).
- `DESCRIBE cad_entities` prints the table schema.
- The layer count runs on `cad_entities` **without a project filter**, so it counts the layers of every project in the database, not only this one. Query `data` instead to see only this drawing.

In [16]:
data = conn.sql("SELECT name, layer, X, Y, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name='PV Toolestown II'")
print(data)
print(conn.execute("DESCRIBE cad_entities").fetchdf())

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM cad_entities
    GROUP BY layer
    ORDER BY count DESC
""").fetchdf()
print(result)

┌──────────────────────────┬─────────────────────────────┬───────────┬──────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│           name           │            layer            │     X     │    Y     │  start_x  │  start_y  │   end_x   │   end_y   │  length   │
│         varchar          │           varchar           │   float   │  float   │   float   │   float   │   float   │   float   │   float   │
├──────────────────────────┼─────────────────────────────┼───────────┼──────────┼───────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF422-2_string_tilt_15° │ EMF_em_PCS_06               │ 694564.44 │ 734915.0 │      NULL │      NULL │      NULL │      NULL │      NULL │
│ EMF422-2_string_tilt_15° │ EMF_em_PCS_06               │ 694552.75 │ 734915.0 │      NULL │      NULL │      NULL │      NULL │      NULL │
│ EMF422-2_string_tilt_15° │ EMF_em_PCS_06               │  694541.0 │ 734915.0 │      NULL │      NULL │      NULL │      NULL │      NULL │
│ EMF4

STRUCTURE DATA

The next two cells count the fixed structures from the string blocks.

Selecting the string blocks
=
Every block reference named exactly `EMF422-2_string_tilt_15°` (one per string). Only the layer (`EMF_em_PCS_<nn>`) and the insertion point are kept.

In [17]:
structures = conn.sql("SELECT layer, X, Y FROM data WHERE name = 'EMF422-2_string_tilt_15°'")
print(structures)

┌───────────────┬───────────┬───────────┐
│     layer     │     X     │     Y     │
│    varchar    │   float   │   float   │
├───────────────┼───────────┼───────────┤
│ EMF_em_PCS_06 │ 694564.44 │  734915.0 │
│ EMF_em_PCS_06 │ 694552.75 │  734915.0 │
│ EMF_em_PCS_06 │  694541.0 │  734915.0 │
│ EMF_em_PCS_06 │  694529.3 │  734915.0 │
│ EMF_em_PCS_06 │ 694517.56 │  734915.0 │
│ EMF_em_PCS_06 │  694505.9 │  734915.0 │
│ EMF_em_PCS_06 │  694494.1 │  734915.0 │
│ EMF_em_PCS_06 │ 694482.44 │  734915.0 │
│ EMF_em_PCS_06 │  694470.7 │  734915.0 │
│ EMF_em_PCS_06 │  694459.0 │  734915.0 │
│       ·       │      ·    │      ·    │
│       ·       │      ·    │      ·    │
│       ·       │      ·    │      ·    │
│ EMF_em_PCS_10 │  694696.1 │  734598.7 │
│ EMF_em_PCS_09 │  694982.4 │  734863.8 │
│ EMF_em_PCS_11 │ 695282.75 │ 734042.25 │
│ EMF_em_PCS_11 │  695271.0 │ 734042.25 │
│ EMF_em_PCS_11 │ 695282.75 │  734032.5 │
│ EMF_em_PCS_11 │ 695212.75 │  734325.0 │
│ EMF_em_PCS_11 │  695220.9 │  734

Counting the structures
=
- `Line` = insertion Y rounded to the metre: the table row the string belongs to.
- Sort order: `Line` descending, then `X` ascending (row by row, north to south, west to east).
- A new structure starts when:
  1. the row changes (`Line` differs from the previous row), or
  2. the X gap to the previous string is larger than **13 m** (strings of one table are ~11.7 m apart).
- `Group` = cumulative sum of the break flags, so its maximum is the number of structures.
- The count is only printed: unlike `BOQ - Minas do Sol.ipynb`, it isn't stored in a DataFrame or exported to Excel.
- The comment "no C type on the edges" is a leftover from another notebook: there are no structure types here.

In [18]:
structure_df = structures.fetchdf()
structure_df['Line'] = round(structure_df['Y'],0)

structure_df.sort_values(by=["Line", "X"], ascending=[False,True], inplace=True, ignore_index=True)

#Different Lines
condition1 = structure_df["Line"] != structure_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = (abs(structure_df["X"] - structure_df["X"].shift()) > 13)


break_condition = condition1 | condition2

structure_df['Group'] = break_condition.cumsum()
print(f'The number of structures is: {max(structure_df['Group'])}')

The number of structures is: 250


CABLE DATA

The next cells measure the trenches, ducts and cables by layer.

Splitting the cable entities into groups
=
One relation per group, selected from `data` by layer name:
- `ac_cables`: `LIKE 'EMF_bt_vala_btca_%xC'` (AC trenches).
- `ac_access`: `LIKE 'EMF_bt_access_ductbank_%xC'`.
- `dc_cables`: exact match on the positive and negative solar cable layers.
- `dc_ducts`: `LIKE 'EMF_bt_eletroduto_%S'`.
- `dc_trenches`: exactly `EMF_DC_vala`.

Matching is case-sensitive. Layers that match nothing are ignored. There is no watercourse ductbank group in this project.

In [19]:
ac_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_vala_btca_%xC'
""")

ac_access = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_access_ductbank_%xC'
""")

dc_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_cabo_solar_negativo' OR layer = 'EMF_cabo_solar_positivo'
""")

dc_ducts = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_eletroduto_%S'
""")

dc_trenches = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_DC_vala'
""")

QTY. OF AC CABLES (m)
=

Prints the AC trench entities (`vala` = trench). The `NNxC` suffix of the layer is the number of AC circuits in the trench.

In [20]:
print(ac_cables)

┌───────────────────────┬───────────┬───────────┬───────────┬───────────┬────────────┐
│         layer         │  start_x  │  start_y  │   end_x   │   end_y   │   length   │
│        varchar        │   float   │   float   │   float   │   float   │   float    │
├───────────────────────┼───────────┼───────────┼───────────┼───────────┼────────────┤
│ EMF_bt_vala_btca_03xC │ 695264.25 │ 734066.25 │  695254.5 │ 734085.75 │  27.383667 │
│ EMF_bt_vala_btca_02xC │ 695264.25 │ 734046.75 │ 695264.25 │ 734066.25 │  19.508259 │
│ EMF_bt_vala_btca_04xC │  695254.5 │ 734085.75 │  695247.2 │ 734105.25 │    21.5989 │
│ EMF_bt_vala_btca_05xC │  695247.2 │ 734105.25 │  695247.2 │ 734124.75 │       19.5 │
│ EMF_bt_vala_btca_06xC │  695247.2 │ 734124.75 │  695238.6 │  734154.0 │  32.217564 │
│ EMF_bt_vala_btca_07xC │  695238.6 │  734154.0 │  695228.4 │  734173.5 │  25.475609 │
│ EMF_bt_vala_btca_08xC │  695228.4 │  734173.5 │ 695228.44 │  734290.6 │ 124.304756 │
│ EMF_bt_vala_btca_02xC │ 694642.56 │ 73482

AC trench length per layer
=
`SUM(length)` per layer, sorted by layer name. The relation is referenced by its variable name (here written as a string). Goes to the `AC Trenches (m)` sheet.

In [21]:
ac_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_cables' GROUP BY layer ORDER BY layer")
print(ac_cables_group)

┌───────────────────────┬────────────────────┐
│         layer         │    total_length    │
│        varchar        │       double       │
├───────────────────────┼────────────────────┤
│ EMF_bt_vala_btca_01xC │ 1714.2477468252182 │
│ EMF_bt_vala_btca_02xC │  695.6563529968262 │
│ EMF_bt_vala_btca_03xC │  287.3086051940918 │
│ EMF_bt_vala_btca_04xC │   754.080816745758 │
│ EMF_bt_vala_btca_05xC │ 367.32415676116943 │
│ EMF_bt_vala_btca_06xC │ 277.91654872894287 │
│ EMF_bt_vala_btca_07xC │ 210.06758785247803 │
│ EMF_bt_vala_btca_08xC │  345.6844325065613 │
│ EMF_bt_vala_btca_09xC │ 217.54494094848633 │
└───────────────────────┴────────────────────┘



QTY. OF AC DUCTS IN ACCESS (m and un.)
=

Prints the ductbanks where the AC trenches cross an access road.

In [22]:
print(ac_access)

┌─────────────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│            layer            │  start_x  │  start_y  │   end_x   │   end_y   │  length   │
│           varchar           │   float   │   float   │   float   │   float   │   float   │
├─────────────────────────────┼───────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_bt_access_ductbank_07xC │ 694864.25 │ 734640.06 │ 694865.94 │  734634.3 │ 5.9999437 │
│ EMF_bt_access_ductbank_08xC │  694660.5 │ 734681.56 │  694654.5 │ 734681.56 │ 6.0008497 │
│ EMF_bt_access_ductbank_04xC │  694614.8 │  734875.3 │ 694620.44 │ 734867.44 │  9.672549 │
└─────────────────────────────┴───────────┴───────────┴───────────┴───────────┴───────────┘



Access ductbank length per layer
=
`SUM(length)` per layer, sorted by layer name. The relation is referenced by its variable name (here written as a string). Goes to the `AC Ducts Access (m)` sheet.

In [23]:
ac_access_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_access' GROUP BY layer ORDER BY layer")
print(ac_access_group)

┌─────────────────────────────┬───────────────────┐
│            layer            │   total_length    │
│           varchar           │      double       │
├─────────────────────────────┼───────────────────┤
│ EMF_bt_access_ductbank_04xC │   9.6725492477417 │
│ EMF_bt_access_ductbank_07xC │ 5.999943733215332 │
│ EMF_bt_access_ductbank_08xC │ 6.000849723815918 │
└─────────────────────────────┴───────────────────┘



Access ductbanks: number of units
=
`COUNT(layer)` per layer, sorted by count. Each entity is one road crossing. Goes to the `AC Ducts Access (un.)` sheet.

In [24]:
ac_access_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'ac_access' GROUP BY layer ORDER BY count DESC")
print(ac_access_count)

┌─────────────────────────────┬───────┐
│            layer            │ count │
│           varchar           │ int64 │
├─────────────────────────────┼───────┤
│ EMF_bt_access_ductbank_08xC │     1 │
│ EMF_bt_access_ductbank_07xC │     1 │
│ EMF_bt_access_ductbank_04xC │     1 │
└─────────────────────────────┴───────┘



QTY. OF DC CABLES (m)
=

Prints every positive and negative solar cable run. Several identical lines on top of each other (same start, end and length) are all summed, so each one must stand for a separate cable.

In [25]:
print(dc_cables)

┌─────────────────────────┬───────────┬──────────┬──────────┬───────────┬─────────┐
│          layer          │  start_x  │ start_y  │  end_x   │   end_y   │ length  │
│         varchar         │   float   │  float   │  float   │   float   │  float  │
├─────────────────────────┼───────────┼──────────┼──────────┼───────────┼─────────┤
│ EMF_cabo_solar_positivo │ 694737.44 │ 735158.0 │ 694744.7 │  735200.0 │ 51.9032 │
│ EMF_cabo_solar_positivo │ 694739.44 │ 735168.0 │ 694744.7 │  735200.0 │  37.244 │
│ EMF_cabo_solar_positivo │ 694739.44 │ 735168.0 │ 694744.7 │  735200.0 │  37.244 │
│ EMF_cabo_solar_positivo │ 694739.44 │ 735168.0 │ 694744.7 │  735200.0 │  37.244 │
│ EMF_cabo_solar_positivo │ 694735.94 │ 735178.0 │ 694744.7 │  735200.0 │   30.75 │
│ EMF_cabo_solar_positivo │ 694735.94 │ 735178.0 │ 694744.7 │  735200.0 │   30.75 │
│ EMF_cabo_solar_positivo │ 694735.94 │ 735187.0 │ 694744.7 │  735200.0 │   21.75 │
│ EMF_cabo_solar_positivo │ 694735.94 │ 735187.0 │ 694744.7 │  735200.0 │   

DC cable length per layer and total
=
- Per-layer `SUM(length)` for the positive and the negative layers.
- The grand total (both polarities) is a second query, rounded to 3 decimals. Only the per-layer table is exported (`DC Cables (m)`).
- Overlapping identical lines are all counted: nothing removes duplicates, so every cable drawn on top of another one is summed again.

In [26]:
print(conn.sql('SELECT * FROM dc_cables ORDER BY length DESC LIMIT 1'))

┌─────────────────────────┬──────────┬──────────┬──────────┬──────────┬─────────┐
│          layer          │ start_x  │ start_y  │  end_x   │  end_y   │ length  │
│         varchar         │  float   │  float   │  float   │  float   │  float  │
├─────────────────────────┼──────────┼──────────┼──────────┼──────────┼─────────┤
│ EMF_cabo_solar_positivo │ 694382.2 │ 734915.0 │ 694549.1 │ 734908.3 │ 214.625 │
└─────────────────────────┴──────────┴──────────┴──────────┴──────────┴─────────┘



Longest DC cable
=
Sanity check only: prints the 1 longest DC cable run (`ORDER BY length DESC LIMIT 1`), to spot a cable that was drawn to the wrong inverter/combiner. Nothing is exported.

In [27]:
dc_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_cables' GROUP BY layer ORDER BY layer")
print(dc_cables_group)

print(f'The total length of DC cables is {round(conn.sql("SELECT SUM(length) FROM 'dc_cables'").fetchone()[0],3)} meters')

┌─────────────────────────┬────────────────────┐
│          layer          │    total_length    │
│         varchar         │       double       │
├─────────────────────────┼────────────────────┤
│ EMF_cabo_solar_negativo │ 122770.94476938248 │
│ EMF_cabo_solar_positivo │ 122770.94476938248 │
└─────────────────────────┴────────────────────┘

The total length of DC cables is 245541.89 meters


QTY. OF DC DUCTS (m and un.)
=

Prints the DC duct entities.

In [28]:
print(dc_ducts)

┌──────────────────────┬───────────┬──────────┬──────────┬───────────┬───────────┐
│        layer         │  start_x  │ start_y  │  end_x   │   end_y   │  length   │
│       varchar        │   float   │  float   │  float   │   float   │   float   │
├──────────────────────┼───────────┼──────────┼──────────┼───────────┼───────────┤
│ EMF_bt_eletroduto_1S │  694815.9 │ 735242.0 │ 694744.7 │ 735200.25 │ 138.25757 │
│ EMF_bt_eletroduto_1S │  694818.4 │ 735232.0 │ 694744.7 │ 735200.25 │ 125.46169 │
│ EMF_bt_eletroduto_1S │ 694820.94 │ 735222.0 │ 694744.7 │ 735200.25 │ 112.51334 │
│ EMF_bt_eletroduto_1S │  694813.3 │ 735213.0 │ 694744.7 │ 735200.25 │    81.375 │
│ EMF_bt_eletroduto_1S │ 694743.56 │ 735206.9 │ 694744.7 │ 735200.25 │    7.7803 │
│ EMF_bt_eletroduto_1S │ 694743.56 │ 735197.0 │ 694744.7 │  735200.0 │     4.125 │
│ EMF_bt_eletroduto_1S │  694736.1 │ 735158.0 │ 694744.7 │  735200.0 │   50.5766 │
│ EMF_bt_eletroduto_1S │ 694725.75 │ 735148.0 │ 694720.4 │  735132.0 │    21.375 │
│ EM

DC duct length per layer
=
Per-layer sum for `1S` / `2S` / `3S` (the suffix is the number of conduits/strings in the duct). Goes to the `DC Ducts (m)` sheet.

In [29]:
dc_ducts_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_ducts' GROUP BY layer ORDER BY layer")
print(dc_ducts_group)

┌──────────────────────┬────────────────────┐
│        layer         │    total_length    │
│       varchar        │       double       │
├──────────────────────┼────────────────────┤
│ EMF_bt_eletroduto_1S │   2919.29461812973 │
│ EMF_bt_eletroduto_2S │ 3147.4713802337646 │
│ EMF_bt_eletroduto_3S │  7563.374762058258 │
└──────────────────────┴────────────────────┘



DC ducts: number of units
=
Number of duct entities per layer and in total. Each LINE/LWPOLYLINE on a duct layer counts as one duct. Goes to the `DC Ducts (un.)` sheet.

In [30]:
dc_ducts_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'dc_ducts' GROUP BY layer ORDER BY count DESC")
print(dc_ducts_count)

print(f'The total quantity of DC ducts is {conn.sql("SELECT COUNT(layer) FROM 'dc_ducts'").fetchone()[0]}')

┌──────────────────────┬───────┐
│        layer         │ count │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ EMF_bt_eletroduto_3S │   617 │
│ EMF_bt_eletroduto_2S │   137 │
│ EMF_bt_eletroduto_1S │   131 │
└──────────────────────┴───────┘

The total quantity of DC ducts is 885


QTY. OF DC TRENCHES (m)
=

Prints the DC trench entities.

In [31]:
print(dc_trenches)

┌─────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│    layer    │  start_x  │  start_y  │   end_x   │   end_y   │  length   │
│   varchar   │   float   │   float   │   float   │   float   │   float   │
├─────────────┼───────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_DC_vala │ 695109.75 │ 734672.75 │  695089.6 │  734667.7 │ 20.729095 │
│ EMF_DC_vala │  695466.8 │  734303.3 │  695443.7 │ 734269.25 │ 41.168777 │
│ EMF_DC_vala │ 695443.25 │  734203.1 │ 695443.25 │  734205.7 │ 2.5771523 │
│ EMF_DC_vala │  694819.9 │  735291.0 │  694829.6 │  735253.3 │ 38.907234 │
│ EMF_DC_vala │  694815.8 │  735281.0 │  694822.5 │  735281.0 │ 6.6574416 │
│ EMF_DC_vala │  694818.9 │  735271.0 │ 694825.06 │  735271.0 │  6.178383 │
│ EMF_DC_vala │ 694821.75 │  735261.0 │  694827.6 │  735261.0 │ 5.8657246 │
│ EMF_DC_vala │ 694825.44 │  735252.0 │ 694829.56 │  735252.0 │   4.13043 │
│ EMF_DC_vala │  694815.9 │  735242.0 │  694829.2 │  735242.0 │ 13.313313 │
│ EMF_DC_val

DC trench length and total
=
Sum of `length` on `EMF_DC_vala` (one row, since there is a single trench layer), plus the grand total. Goes to the sheet named `DC Trenches (un.)`, although the value is a length in metres.

In [32]:
dc_trenches_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_trenches' GROUP BY layer ORDER BY layer")
print(dc_trenches_group)

print(f'\nThe total length of DC trenches is {round(conn.sql("SELECT SUM(length) FROM 'dc_trenches'").fetchone()[0],3)} meters')

┌─────────────┬────────────────────┐
│    layer    │    total_length    │
│   varchar   │       double       │
├─────────────┼────────────────────┤
│ EMF_DC_vala │ 3690.4556112512946 │
└─────────────┴────────────────────┘


The total length of DC trenches is 3690.456 meters


OVERVIEW
=

Writes every summary to its own sheet of `C:\Users\Usuario\Desktop\Toolestown II - BOQ.xlsx` (hardcoded path, overwritten on every run), then closes the DuckDB connection.
- The number of structures isn't exported.
- The `DC Trenches (un.)` sheet holds a length in metres, not units.

In [33]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\Toolestown II - BOQ.xlsx") as xlsx:
    ac_cables_group.df().to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    ac_access_group.df().to_excel(xlsx, sheet_name='AC Ducts Access (m)', index=False)
    ac_access_count.df().to_excel(xlsx, sheet_name='AC Ducts Access (un.)', index=False)
    dc_cables_group.df().to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group.df().to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_count.df().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group.df().to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)


conn.close()
